# EEG-Based Epileptic Seizure Detection
**SSIT 2025-26 | CHB-MIT Dataset | SVM Classifier**

Run cells top to bottom. Do not skip any cell.

## Cell 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install mne scikit-learn imbalanced-learn pandas matplotlib seaborn joblib --quiet

## Cell 2 — Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mne
mne.set_log_level('WARNING')

from scipy.stats import skew, kurtosis
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc
)
from imblearn.over_sampling import SMOTE

print('All imports successful.')

## Cell 3 — Configuration

**Edit `DATA_DIR` to match where your chb01 folder is.**

In [ ]:
# ── EDIT THIS PATH ──────────────────────────────────────────────
DATA_DIR    = 'chbmit/chb01'        # folder containing .edf files
SUBJECT     = 'chb01'
SUMMARY_FILE = os.path.join(DATA_DIR, 'chb01-summary.txt')
# ────────────────────────────────────────────────────────────────

SEG_LEN     = 12      # seconds per segment
CHANNEL     = 0       # which EEG channel to use (0 = first)
SFREQ       = 256     # expected sampling rate
RANDOM_SEED = 42

MODEL_DIR   = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Data directory : {DATA_DIR}')
print(f'Summary file   : {SUMMARY_FILE}')
print(f'Models saved to: {MODEL_DIR}')

## Cell 4 — Parse seizure annotations from summary file

In [ ]:
def parse_summary(summary_path):
    """
    Reads chb01-summary.txt and returns a dict:
        {'chb01_03.edf': [(2996, 3036)], 'chb01_04.edf': [(1467, 1494)], ...}
    Files with 0 seizures are not included (default to empty list via .get).
    """
    seizure_map = {}
    current_file = None
    starts = []

    with open(summary_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('File Name:'):
                current_file = line.split(':', 1)[-1].strip()
                starts = []
            elif line.startswith('Seizure Start Time:'):
                val = line.split(':', 1)[-1].replace('seconds', '').strip()
                starts.append(int(val))
            elif line.startswith('Seizure End Time:'):
                val = line.split(':', 1)[-1].replace('seconds', '').strip()
                end = int(val)
                if current_file not in seizure_map:
                    seizure_map[current_file] = []
                seizure_map[current_file].append((starts[-1], end))

    return seizure_map


summary = parse_summary(SUMMARY_FILE)

print('Files with seizures:')
for fname, seizes in summary.items():
    print(f'  {fname}: {seizes}')

## Cell 5 — Segment EEG and assign labels

In [ ]:
def segment_eeg(data, sfreq, seizures, seg_len=12, channel=0):
    """
    Splits a single-subject EEG recording into non-overlapping
    windows of `seg_len` seconds and labels each as 0 or 1.

    Parameters
    ----------
    data     : np.ndarray  shape (n_channels, n_samples)
    sfreq    : float       sampling frequency in Hz
    seizures : list        [(start_sec, end_sec), ...]
    seg_len  : int         window length in seconds
    channel  : int         which channel to use

    Returns
    -------
    X : np.ndarray  shape (n_segments, seg_len * sfreq)
    y : np.ndarray  shape (n_segments,)  values 0 or 1
    """
    seg_samples = int(seg_len * sfreq)
    signal      = data[channel]
    n_segments  = len(signal) // seg_samples
    X, y = [], []

    for i in range(n_segments):
        start_s = i * seg_len
        end_s   = start_s + seg_len
        segment = signal[i * seg_samples : (i + 1) * seg_samples]

        label = 0
        for sz_start, sz_end in seizures:
            # label = 1 if any overlap with a seizure window
            if not (end_s <= sz_start or start_s >= sz_end):
                label = 1
                break

        X.append(segment)
        y.append(label)

    return np.array(X), np.array(y)

## Cell 6 — Load all EDF files and build dataset

In [ ]:
all_X, all_y = [], []
skipped = []

edf_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.edf')])
print(f'Found {len(edf_files)} EDF files.\n')

for edf_file in edf_files:
    edf_path = os.path.join(DATA_DIR, edf_file)
    seizures = summary.get(edf_file, [])

    try:
        raw  = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        data = raw.get_data()
        sfreq_actual = raw.info['sfreq']

        if sfreq_actual != SFREQ:
            print(f'  [SKIP] {edf_file} — unexpected sfreq {sfreq_actual}')
            skipped.append(edf_file)
            continue

        X_file, y_file = segment_eeg(data, sfreq_actual, seizures,
                                     seg_len=SEG_LEN, channel=CHANNEL)
        all_X.append(X_file)
        all_y.append(y_file)

        status = f'{y_file.sum()} seizure segs' if y_file.sum() > 0 else 'no seizures'
        print(f'  {edf_file}: {len(y_file)} segments ({status})')

    except Exception as e:
        print(f'  [ERROR] {edf_file}: {e}')
        skipped.append(edf_file)

X_raw = np.concatenate(all_X, axis=0)
y     = np.concatenate(all_y, axis=0)

print(f'\nTotal segments : {len(y)}')
print(f'Non-seizure (0): {(y == 0).sum()}')
print(f'Seizure     (1): {(y == 1).sum()}')
print(f'Skipped files  : {skipped}')

## Cell 7 — Feature extraction

In [ ]:
def extract_time_features(segment):
    """
    Extracts 6 time-domain statistical features from one EEG segment.
    Returns a list: [mean, std, skewness, kurtosis, zcr, rms]
    """
    mean  = np.mean(segment)
    std   = np.std(segment)
    skewn = skew(segment)
    kurt  = kurtosis(segment)
    zcr   = np.sum(np.diff(np.sign(segment)) != 0) / (len(segment) - 1)
    rms   = np.sqrt(np.mean(segment ** 2))
    return [mean, std, skewn, kurt, zcr, rms]


feature_matrix = np.array([extract_time_features(seg) for seg in X_raw])

feature_names = ['Mean', 'Std Dev', 'Skewness', 'Kurtosis', 'ZCR', 'RMS']
df_features = pd.DataFrame(feature_matrix, columns=feature_names)
df_features['Label'] = y

print(f'Feature matrix shape: {feature_matrix.shape}')
print(df_features.describe().round(4))

## Cell 8 — Visualise features (seizure vs non-seizure)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, feat in enumerate(feature_names):
    for label, color, name in [(0, '#4C9BE8', 'Non-seizure'), (1, '#E84C4C', 'Seizure')]:
        axes[i].hist(
            df_features[df_features['Label'] == label][feat],
            bins=40, alpha=0.6, color=color, label=name, density=True
        )
    axes[i].set_title(feat, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')

plt.suptitle('Feature Distributions: Seizure vs Non-Seizure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_distributions.png')

## Cell 9 — Scale features + handle class imbalance with SMOTE

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(feature_matrix)

print(f'Before SMOTE — class counts: {np.unique(y, return_counts=True)}')

sm = SMOTE(random_state=RANDOM_SEED)
X_bal, y_bal = sm.fit_resample(X_scaled, y)

print(f'After  SMOTE — class counts: {np.unique(y_bal, return_counts=True)}')

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    stratify=y_bal,
    random_state=RANDOM_SEED
)

print(f'\nTrain size: {len(y_train)} | Test size: {len(y_test)}')

## Cell 10 — Train SVM with GridSearchCV

In [ ]:
param_grid = {
    'C'     : [0.1, 1, 10, 100],
    'gamma' : ['scale', 0.1, 0.01, 0.001]
}

grid = GridSearchCV(
    SVC(kernel='rbf', class_weight='balanced', probability=True),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(f'\nBest parameters : {grid.best_params_}')
print(f'Best CV F1-score: {grid.best_score_:.4f}')

## Cell 11 — Evaluate on test set

In [ ]:
best_model = grid.best_estimator_
y_pred     = best_model.predict(X_test)
y_prob     = best_model.predict_proba(X_test)[:, 1]

print('=' * 50)
print('CLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(y_test, y_pred,
      target_names=['Non-Seizure', 'Seizure']))

## Cell 12 — Confusion matrix plot

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Confusion matrix ---
disp = ConfusionMatrixDisplay(cm, display_labels=['Non-Seizure', 'Seizure'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold')

# --- ROC curve ---
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#E84C4C', lw=2,
             label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_evaluation.png')

## Cell 13 — Save model and scaler

In [ ]:
joblib.dump(best_model, os.path.join(MODEL_DIR, 'svm_model.pkl'))
joblib.dump(scaler,     os.path.join(MODEL_DIR, 'scaler.pkl'))

print('Saved models/svm_model.pkl')
print('Saved models/scaler.pkl')

## Cell 14 — Predict on a new EDF file

Use this cell to test on any new recording after training.

In [ ]:
def predict_edf(edf_path, model_path='models/svm_model.pkl',
                scaler_path='models/scaler.pkl',
                seg_len=12, channel=0):
    """
    Loads a saved model and scaler, reads an EDF file,
    and returns a DataFrame with per-segment predictions.
    """
    model  = joblib.load(model_path)
    scaler = joblib.load(scaler_path)

    raw    = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    data   = raw.get_data()
    sfreq  = raw.info['sfreq']

    seg_samples = int(seg_len * sfreq)
    signal      = data[channel]
    n_segments  = len(signal) // seg_samples

    results = []
    for i in range(n_segments):
        seg  = signal[i * seg_samples : (i + 1) * seg_samples]
        feat = np.array([extract_time_features(seg)])
        feat_scaled = scaler.transform(feat)
        pred = model.predict(feat_scaled)[0]
        prob = model.predict_proba(feat_scaled)[0][1]
        results.append({
            'Segment'   : i + 1,
            'Start (s)' : i * seg_len,
            'End (s)'   : (i + 1) * seg_len,
            'Prediction': 'SEIZURE' if pred == 1 else 'Normal',
            'Confidence': f'{prob:.2%}'
        })

    df = pd.DataFrame(results)
    print(f'File: {os.path.basename(edf_path)}')
    print(f'Total segments: {len(df)}')
    print(f'Seizure segments detected: {(df["Prediction"] == "SEIZURE").sum()}')
    return df


# ── Change this to any EDF file you want to test ──
TEST_EDF = os.path.join(DATA_DIR, 'chb01_04.edf')

results_df = predict_edf(TEST_EDF)
results_df[results_df['Prediction'] == 'SEIZURE']

## Cell 15 — Streamlit app (run separately)

Save the cell below as `app.py` and run: `streamlit run app.py`

In [ ]:
app_code = '''
import streamlit as st
import mne, numpy as np, pandas as pd, joblib, tempfile, os
from scipy.stats import skew, kurtosis

st.set_page_config(page_title="EEG Seizure Detector", page_icon="🧠", layout="wide")
st.title("🧠 EEG Seizure Detection")
st.caption("CHB-MIT dataset · SVM classifier · SSIT 2025-26")

@st.cache_resource
def load_model():
    model  = joblib.load("models/svm_model.pkl")
    scaler = joblib.load("models/scaler.pkl")
    return model, scaler

def extract_features(segment):
    mean  = np.mean(segment)
    std   = np.std(segment)
    skewn = skew(segment)
    kurt  = kurtosis(segment)
    zcr   = np.sum(np.diff(np.sign(segment)) != 0) / (len(segment) - 1)
    rms   = np.sqrt(np.mean(segment ** 2))
    return [mean, std, skewn, kurt, zcr, rms]

model, scaler = load_model()

uploaded = st.file_uploader("Upload an EDF file", type=["edf"])

if uploaded:
    with tempfile.NamedTemporaryFile(suffix=".edf", delete=False) as tmp:
        tmp.write(uploaded.read())
        tmp_path = tmp.name

    with st.spinner("Reading EDF file..."):
        raw   = mne.io.read_raw_edf(tmp_path, preload=True, verbose=False)
        data  = raw.get_data()
        sfreq = raw.info["sfreq"]
        seg_samples = int(12 * sfreq)
        signal = data[0]
        n_segs = len(signal) // seg_samples

    rows = []
    for i in range(n_segs):
        seg  = signal[i*seg_samples:(i+1)*seg_samples]
        feat = np.array([extract_features(seg)])
        pred = model.predict(scaler.transform(feat))[0]
        prob = model.predict_proba(scaler.transform(feat))[0][1]
        rows.append({
            "Segment"   : i + 1,
            "Start (s)" : i * 12,
            "End (s)"   : (i + 1) * 12,
            "Prediction": "🔴 SEIZURE" if pred == 1 else "🟢 Normal",
            "Confidence": f"{prob:.1%}"
        })
    os.unlink(tmp_path)

    df = pd.DataFrame(rows)
    seizure_count = (df["Prediction"].str.contains("SEIZURE")).sum()

    col1, col2, col3 = st.columns(3)
    col1.metric("Total segments", n_segs)
    col2.metric("Seizure segments", seizure_count)
    col3.metric("Recording length", f"{n_segs * 12}s")

    st.dataframe(
        df.style.apply(
            lambda x: ["background-color: #ffe0e0" if "SEIZURE" in str(v) else "" for v in x],
            axis=1
        ),
        use_container_width=True
    )
    st.download_button("Download results as CSV",
                       df.to_csv(index=False), "results.csv", "text/csv")
'''

with open('app.py', 'w') as f:
    f.write(app_code.strip())

print('app.py saved. Run with: streamlit run app.py')